### Verben – Lösung Schritt für Schritt

*31. Bundeswettbewerb Informatik (2012), 1. Runde, Junioraufgabe 2*

Im Deutschunterricht wird gerade die Konjugation deutscher Verben behandelt. Mehmet denkt sich dabei: „Wozu braucht man da
überhaupt einen Deutschlehrer? Das sind doch ganz einfache Regeln. Die programmiere ich mal fix; dann kann mein Smartphone
hoffentlich die Hausaufgaben der nächsten Wochen erledigen.“ Nach einiger Zeit muss er aber zugeben, dass sein Programm immer
noch nicht fertig ist und dass er nach fast jeder Deutschstunde irgendwelche kniffligen Fälle mit seinem Lehrer bespricht.

#### Junioraufgabe 2

Betrachte einen kleinen Teil von dem, was Mehmet vorhat: Schreibe ein Programm, das aus einer beliebigen Form eines deutschen
Verbs durch Anwendung von Regeln versucht, die Grundform (den Infinitiv) des Verbs herauszufinden. Erhält das Programm z. B.
die Eingabe „sagst“, soll es „sagen“ ausgeben.

Dein Programm muss zumindest bei Eingaben wie den folgenden die richtige Ausgabe produzieren:

- sagst
- leitete
- geforscht
- schweigend
- trag

Zeige durch geeignete Eingabe-Ausgabepaare, was dein Programm kann. Zeige auch durch andere (vielleicht lustige) Beispiele,
was es nicht kann.

*Freiwillige Zusatzaufgabe:* Wenn du Lust hast, kannst du probieren, dein Programm so weiter zu entwickeln, dass es sich auch
von hinterhältigen Eingaben wie „gebet“, „arbeite“, „genehmigt“, „gehört“ oder „mitgehalten“ / „mitgestalten“ nicht aus der
Fassung bringen lässt. Das ist aber schwierig; und unregelmäßige Verben muss dein Programm sowieso nicht korrekt behandeln.

Beispieldaten gibt es zu dieser Aufgabe nicht.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Regelbasierte Textverarbeitung:** Wir erkennen eine Verbform an ihren **Endungen** (und am Wortanfang `ge-`) und ersetzen sie durch die Infinitivendung `-en`. Das sind einfache Regeln, die man als Liste von Endungen und ein paar Schritten schreiben kann.
- **Reihenfolge der Regeln:** Manche Endungen sind das Ende anderer (`t` ist das Ende von `st`, `st` das Ende von `test`). Prüft man die kurze Endung zuerst, wird zu wenig abgeschnitten. Deshalb prüft man die **längsten Endungen zuerst**.
- **Strings bearbeiten:** `wort.startswith('ge')`, `wort.endswith(endung)`, Ausschnitte wie `wort[2:]` und `wort[:-3]` (alles bis auf die letzten 3 Zeichen).
- **Grenzen von Regeln:** Mit Regeln allein kann man nicht alle Fälle richtig lösen („gehört“ → „hören“, aber „gehen“?). Die Zusatzaufgabe zeigt, wo das Verfahren versagt.
- **Reguläre Ausdrücke (Fortgeschrittene):** Mit dem Modul `re` lässt sich dieselbe Regel in einer einzigen Zeile ausdrücken (siehe „Für Neugierige“).

**Nützliche Funktionen und Bibliotheken**
- `startswith`, `endswith`, `lower`, Ausschnitte, Listen mit `for`, `import re` mit `re.sub` (nur für die Erweiterung am Ende)

## Die Idee

Die Verbformen des Verbs „sagen“:

| | Präsens | Präteritum | Konjunktiv I | Konjunktiv II |
|---|---|---|---|---|
| ich | sage | sagte | sage | sagte |
| du | sagst | sagtest | sagest | sagtest |
| er/sie/es | sagt | sagte | sage | sagte |
| wir | sagen | sagten | sagen | sagten |
| ihr | sagt | sagtet | saget | sagtet |
| sie | sagen | sagten | sagen | sagten |

Dazu kommen die Partizipien „sagend“ und „gesagt“ und der Imperativ „sag“ / „sage“ / „sagt“. Der **Stamm** ist immer „sag“. Nach dem Stamm steht eine der Endungen (oder keine). Wir schneiden die Endung ab und hängen `en` an.

Die Endungen, nach Länge sortiert: `test`, `tet`, `ten`, `te`, `st`, `nd`, `t`, `n`. Beim Verb „leiten“ steht vor der Endung oft noch ein `e` (`leitete`, `leiteten`). Das `e` wird mit entfernt. Das Partizip beginnt mit `ge`, das auch entfällt.

**Das Verfahren:**
1. Beginnt das Wort mit `ge` (und ist danach noch genug übrig), schneide `ge` ab.
2. Prüfe die Endungen der Reihe nach (**längste zuerst**). Endet das Wort auf eine Endung, schneide sie ab und höre auf.
3. Endet das Ergebnis auf `e`, schneide dieses `e` ab.
4. Hänge `en` an.

## Schritt 1: Die Regeln als Daten

Die Endungen stehen in einer Liste, die **nach Länge sortiert** ist (längste zuerst). Dann genügt eine Schleife, die die Endungen der Reihe nach probiert. Die Reihenfolge bei gleicher Länge ist egal, weil nur eine davon passen kann.
Warum die Reihenfolge wichtig ist, zeigt ein Beispiel: „sagtest“ endet auf `test`, aber auch auf `st` und `t`. Würden wir zuerst `t` prüfen, bliebe „sagtes“ übrig und das Ergebnis wäre „sagtesen“.

In [ ]:
ENDUNGEN = ['test', 'tet', 'ten', 'te', 'st', 'nd', 't', 'n']       # längste zuerst


def entferne_endung(wort):
    for endung in ENDUNGEN:
        if wort.endswith(endung):
            return wort[:-len(endung)]          # alles bis auf die letzten Zeichen
    return wort


print(entferne_endung('sagtest'), entferne_endung('sagst'), entferne_endung('sagt'), entferne_endung('trag'))
# Was würde passieren, wenn man die kurze Endung 't' zuerst prüft?
falsch = 'sagtest'[:-1]
print(falsch, '(falsch: so bliebe "sagtes" und am Ende "sagtesen")')

sag sag sag trag
sagtes (falsch: so bliebe "sagtes" und am Ende "sagtesen")


## Schritt 2: Das ganze Verfahren

`grundform(wort)` führt die vier Schritte aus. Wir machen zuerst alles klein (`lower()`).

- Schritt 1: `ge` am Anfang entfernen, aber nur, wenn danach noch mindestens 3 Buchstaben übrig bleiben (sonst würde aus „gehen“ zu wenig). Hier prüfen wir einfach `len(wort) > 4`.
- Schritt 2: Endung abschneiden.
- Schritt 3: Ein `e` am Ende abschneiden („leite“ → „leit“).
- Schritt 4: `en` anhängen.

In [ ]:
def grundform(wort):
    wort = wort.lower()
    if wort.startswith('ge') and len(wort) > 4:
        wort = wort[2:]                         # Partizip: ge- weg
    wort = entferne_endung(wort)
    if wort.endswith('e'):
        wort = wort[:-1]                        # leit-e-te -> leit
    return wort + 'en'


for form in ['sagst', 'leitete', 'geforscht', 'schweigend', 'trag']:
    print(f'{form:12} -> {grundform(form)}')

sagst        -> sagen
leitete      -> leiten
geforscht    -> forschen
schweigend   -> schweigen
trag         -> tragen


Alle fünf Pflichtbeispiele der Aufgabe werden richtig gelöst: sagen, leiten, forschen, schweigen, tragen.

## Schritt 3: Weitere Beispiele

Zuerst die Verbformen von „sagen“ aus der Tabelle oben und weitere Formen anderer Verben, die alle Endungen abdecken (aus den Lösungshinweisen): „glaubtest“, „meintet“, „tippten“, „holen“.

In [ ]:
sagen_formen = ['sage', 'sagst', 'sagt', 'sagen', 'sagte', 'sagtest', 'sagten', 'sagtet', 'sagest', 'saget', 'sagend', 'gesagt', 'sag']
print([grundform(f) for f in sagen_formen])

for form in ['glaubtest', 'meintet', 'tippten', 'holen', 'leiteten', 'geleitet', 'arbeitest', 'lachend']:
    print(f'{form:12} -> {grundform(form)}')

['sagen', 'sagen', 'sagen', 'sagen', 'sagen', 'sagen', 'sagen', 'sagen', 'sagen', 'sagen', 'sagen', 'sagen', 'sagen']
glaubtest    -> glauben
meintet      -> meinen
tippten      -> tippen
holen        -> holen
leiteten     -> leiten
geleitet     -> leien
arbeitest    -> arbeien
lachend      -> lachen


Alle Formen von „sagen“ werden zu „sagen“, auch Konjunktiv (`sagest`, `saget`) und Partizip (`gesagt`). Auch „leiteten“, „glaubtest“, „meintet“ und „lachend“ klappen.
Nicht richtig sind „geleitet“ (`leien` statt `leiten`) und „arbeitest“ (`arbeien` statt `arbeiten`): Der Stamm endet hier selbst auf `t` (leit, arbeit), und das Programm hält das `t` oder `et` für eine Endung. Dazu mehr in Schritt 4.

## Schritt 4: Was das Programm nicht kann

Die Zusatzaufgabe nennt „hinterhältige“ Eingaben. Die Regeln sind auf Endungen und das `ge-` am Anfang beschränkt, das Programm kennt die Wörter selbst nicht. Wir zeigen seine Grenzen:

In [ ]:
for form in ['gebet', 'arbeite', 'genehmigt', 'gehört', 'mitgehalten', 'mitgestalten', 'kichern', 'schreite', 'geleitet', 'arbeitest', 'gehen', 'ging']:
    print(f'{form:14} -> {grundform(form)}')

gebet          -> ben
arbeite        -> arbeien
genehmigt      -> nehmigen
gehört         -> hören
mitgehalten    -> mitgehalen
mitgestalten   -> mitgestalen
kichern        -> kicheren
schreite       -> schreien
geleitet       -> leien
arbeitest      -> arbeien
gehen          -> hen
ging           -> gingen


Das Verhalten im Einzelnen:

| Eingabe | Ausgabe | Richtig wäre | Warum der Fehler |
|---|---|---|---|
| gebet | ben | beten | Das `ge` gehört zum Wort (kein Partizip) und wird abgeschnitten. |
| arbeite | arbeien | arbeiten | Das `t` gehört zum Stamm, wird aber als Endung `te` abgeschnitten. |
| genehmigt | nehmigen | genehmigen | `ge` gehört zum Wort „genehmigen“. |
| gehört | hören | hören | Stimmt, wenn „gehört“ das Partizip von „hören“ ist. Von „gehören“ (er gehört) wäre es falsch. |
| mitgehalten | mitgehalen | mithalten | `ge` steht mitten im Wort, die Endung `ten` ist Teil des Stamms. |
| mitgestalten | mitgestalen | mitgestalten | dasselbe |
| kichern | kicheren | kichern | Der Infinitiv endet nicht auf `-en`, sondern auf `-ern`. |
| schreite | schreien | schreiten | Endung `te` abgeschnitten, aber das `t` gehört zum Stamm. |
| geleitet | leien | leiten | Der Stamm „leit“ endet auf `t`, das Programm schneidet `tet` ab. |
| arbeitest | arbeien | arbeiten | Der Stamm „arbeit“ endet auf `t`, das Programm schneidet `test` ab. |
| gehen | hen | gehen | `ge` gehört zum Stamm, wird aber als Vorsilbe abgeschnitten. |
| ging | gingen | gehen | Unregelmäßige Verben kennt das Programm nicht. |

Diese Fälle lassen sich nicht allgemein lösen. Das Programm bräuchte ein **Wörterbuch**, in dem es nachschaut, ob „arbeit“ oder „arbeite“ ein Verbstamm ist. Mit reinen Regeln ist zum Beispiel „verzehrst“ (richtig: verzehren)
nicht von „kicherst“ (richtig: kichern) zu unterscheiden.

## Für Neugierige: Dieselbe Regel mit einem regulären Ausdruck

Ein **regulärer Ausdruck** beschreibt ein Textmuster. Mit `re.sub(muster, ersatz, text)` ersetzt Python Textstellen, die zum Muster passen. Die Lösungshinweise geben diesen Ausdruck an:

`^(ge)?(.*?)e?(test|st|te|tet|t|ten|n|nd)?$`

- `^` und `$`: Anfang und Ende des Wortes.
- `(ge)?`: ein mögliches `ge` am Anfang. `?` heißt: „vielleicht“.
- `(.*?)`: der **Stamm**, möglichst kurz (das zweite `?`), damit die Endungen vollständig abgeschnitten werden. Das ist die zweite Klammer, im Ersatz `\2`.
- `e?`: ein mögliches `e`.
- `(test|st|...)?`: eine mögliche Endung.

Der Ersatz `\2en` setzt den Stamm (zweite Klammer) mit `en` zusammen. Wir vergleichen die beiden Verfahren an allen Wörtern von oben.

In [ ]:
import re

MUSTER = r'^(ge)?(.*?)e?(test|st|te|tet|t|ten|n|nd)?$'


def grundform_regex(wort):
    return re.sub(MUSTER, r'\2en', wort.lower())


woerter = ['sagst', 'leitete', 'geforscht', 'schweigend', 'trag'] + sagen_formen + \
          ['glaubtest', 'meintet', 'tippten', 'holen', 'leiteten', 'geleitet', 'arbeitest', 'lachend']
unterschiede = []
for wort in woerter:
    if grundform(wort) != grundform_regex(wort):
        unterschiede.append((wort, grundform(wort), grundform_regex(wort)))
print('Unterschiede zwischen beiden Verfahren:', unterschiede)
for wort in ['gebet', 'arbeite', 'genehmigt', 'gehört', 'mitgehalten', 'kichern']:
    print(f'{wort:12} -> {grundform_regex(wort)}')

Unterschiede zwischen beiden Verfahren: []
gebet        -> ben
arbeite      -> arbeien
genehmigt    -> nehmigen
gehört       -> hören
mitgehalten  -> mitgehalen
kichern      -> kicheren


Beide Verfahren liefern fast immer dasselbe. Die regulären Ausdrücke sind kürzer, aber schwerer zu lesen. Für Anfänger ist die Version mit der Liste der Endungen und den einzelnen Schritten besser nachzuvollziehen.